In [1]:
from dolfinx import io, fem, mesh as msh
from Training_utils import train_set
from SPDE_problems import *
from FEniCSx_solver import plot_fn
import torch
import torch_geometric as tg
from SPDE_problems import Data_to_solver
from SUPG_prediction_models import *
from FEniCSx_solver import interpolate_expr
from FEniCSx_PyTorch_interface import fem_solver


mlp, gcn, sage, gat, gatv2 = MLP(), GCN(), SAGE(), GAT(), GATv2()

class batched_loss_fn():
    def __init__(self):
        self.fsl = {}
        for num in range(len(train_set)):
            fs , G = Data_to_solver(num, train = True)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn()

def train(model, loader, optimizer, device):
    #model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        ptr = data.ptr
        idx = data.mesh_id
        out = model(data)
        loss = loss_fn(ptr, idx, out)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)
    

In [9]:
model = gatv2
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer)
loader = train_loader(batch_size=1)
for i in range(1000):
    print(i)
    print(train(gatv2, loader, optimizer, 'cpu'))

0
0.40000749853241113
1
0.39978130428432285
2
0.3995876910918138
3
0.39944944867760446
4
0.3993603442303621
5
0.39914326871006656
6
0.3947081229440114
7
0.37508017528741466
8
0.3777498155401521
9
0.33391772459636243
10
0.3282498443720876
11
0.32394420001643004
12
0.3113410521391829
13
0.30850635388080927
14
0.3245390396382804
15
0.30838723931339534
16
0.29087174676113714
17
0.309762390453319
18
0.31521098148503124
19
0.30863356044865625
20
0.30171188578953495
21
0.29307439293831355
22
0.28312013486079607
23
0.2707972422000363
24
0.258077117779232
25
0.25057017683455096
26
0.244641380337312
27
0.23756064274092833
28
0.23466020450510686
29
0.23057515234092368
30
0.22885841953798677
31
0.23094502214644308
32
0.2907116415245499
33
0.30910712999116186
34
0.30865877961107135
35
0.30818878688581786
36
0.30771161663756175
37
0.3072290680798398
38
0.30674266689077906
39
0.30624676161526376
40
0.3057419453744139
41
0.30523050012352954
42
0.30470912134228456
43
0.3041828356827781
44
0.30364870259

KeyboardInterrupt: 

In [ ]:

torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, "data/models/GATv2_self_supervised.pth")